In [ ]:
import os
import logging
import ast
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# -------------------------------------------------------------------------
# 1. CONFIGURE LOGGING FOR STATUS UPDATES
# -------------------------------------------------------------------------
# By default, logging prints to stdout. You can adjust level to DEBUG for even more detail.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

CHUNKS_DIR = "/mnt/4TBSTORE/Dr_Kabuye4TB/processed_chunks"   # Directory containing CSVs with embeddings
TARGET_COLUMN = "category"
IGNORE_LABEL = -1                 # We might skip rows with this label
TEST_SIZE = 0.2                   # 20% test split
RANDOM_SEED = 42


import numpy as np
import re

def parse_space_separated_floats(embed_string: str) -> list:
    """
    Takes a string like:
      "[-2.86691338e-01 -2.14018077e-01 2.51793355e-01 -1.58184245e-01]"
    and returns a Python list of floats.
    """
    # Remove leading/trailing brackets
    embed_string = embed_string.strip()
    embed_string = embed_string.strip("[]")

    # split on whitespace
    tokens = embed_string.split()
    # convert each token to float
    floats = list(map(float, tokens))
    return floats

# -------------------------------------------------------------------------
# 2. LOAD EMBEDDINGS & LABELS WITH TQDM PROGRESS
# -------------------------------------------------------------------------
def load_embeddings_and_labels(chunks_dir):
    """
    Loads CSVs with 'llm_embeddings' and 'category' columns,
    concatenates them into big arrays X, y.
    """
    logging.info(f"Looking for CSV files in {chunks_dir} ...")
    csv_files = glob(os.path.join(chunks_dir, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {chunks_dir}.")

    all_embeddings = []
    all_labels = []

    # Use TQDM to show a progress bar over CSV files
    for csv_path in tqdm(csv_files, desc="Reading CSVs", unit="file"):
        df = pd.read_csv(csv_path)

        if "llm_embeddings" not in df.columns:
            logging.warning(f"Skipping {csv_path}: missing 'llm_embeddings' column.")
            continue
        if TARGET_COLUMN not in df.columns:
            logging.warning(f"Skipping {csv_path}: missing '{TARGET_COLUMN}' column.")
            continue

        # Convert string representations of Python lists to actual lists
        # raw_embeds = df["llm_embeddings"].apply(lambda x: ast.literal_eval(str(x)))
        # embeddings = np.vstack(raw_embeds.values)  # shape: (n_rows, embedding_dim)
        
        raw_embeds = df["llm_embeddings"].apply(lambda x: parse_space_separated_floats(str(x)))
        embeddings = np.vstack(raw_embeds.values)


        labels = df[TARGET_COLUMN].values

        all_embeddings.append(embeddings)
        all_labels.append(labels)

    # Concatenate all chunks
    X = np.vstack(all_embeddings)
    y = np.concatenate(all_labels)

    logging.info(f"Total samples loaded: {X.shape[0]}, Embedding dimension: {X.shape[1]}")
    return X, y

# -------------------------------------------------------------------------
# 3. MAIN PIPELINE WITH STATUS LOGS
# -------------------------------------------------------------------------
def main():
    logging.info("=== Starting pipeline ===")

    # Step A: Load Data
    logging.info("Step A: Loading embeddings and labels")
    X, y = load_embeddings_and_labels(CHUNKS_DIR)
    logging.info(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

    # Step B: Filter out rows with label == -1, if you need to ignore them
    logging.info("Step B: Filtering out label = -1 (if it indicates unknown/missing)")
    valid_mask = (y != IGNORE_LABEL)
    X = X[valid_mask]
    y = y[valid_mask]
    logging.info(f"After filtering, shape of X: {X.shape}, shape of y: {y.shape}")

    # Step C: Train/Test Split
    logging.info("Step C: Splitting into train/test sets")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )
    logging.info(f"Train set size: {X_train.shape[0]}, Test set size: {X_test.shape[0]}")

    # Step D: Train Classifier
    logging.info("Step D: Initializing RandomForestClassifier")
    clf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_SEED,
        verbose=0  # If you want scikit-learn's own verbose logs per tree, set > 0
    )

    logging.info("         ... Training started ...")
    clf.fit(X_train, y_train)
    logging.info("         ... Training complete ...")

    # Step E: Predictions & Metrics
    logging.info("Step E: Evaluating on test set")
    y_pred = clf.predict(X_test)

    # Accuracy, Precision, Recall, F1
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="binary")
    rec = recall_score(y_test, y_pred, average="binary")
    f1 = f1_score(y_test, y_pred, average="binary")

    logging.info(f"Accuracy : {acc:.4f}")
    logging.info(f"Precision: {prec:.4f}")
    logging.info(f"Recall   : {rec:.4f}")
    logging.info(f"F1-score : {f1:.4f}")

    logging.info("Confusion Matrix:")
    logging.info("\n" + str(confusion_matrix(y_test, y_pred)))

    logging.info("Classification Report:")
    logging.info("\n" + classification_report(y_test, y_pred))

    logging.info("=== Pipeline finished ===")

if __name__ == "__main__":
    main()


2025-03-24 14:38:31,845 | INFO | === Starting pipeline ===
2025-03-24 14:38:31,846 | INFO | Step A: Loading embeddings and labels
2025-03-24 14:38:31,847 | INFO | Looking for CSV files in /mnt/4TBSTORE/Dr_Kabuye4TB/processed_chunks ...
Reading CSVs:   1%|▏         | 6/443 [03:23<4:08:47, 34.16s/file]/tmp/ipykernel_230278/3932804370.py:69: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)
Reading CSVs:   7%|▋         | 33/443 [19:41<4:15:13, 37.35s/file]

In [8]:
import os
import ast
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# -------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------
TRAIN_DIR = "/mnt/4TBSTORE/Dr_Kabuye4TB/processed_chunks"   # CSVs for training in chunks
TEST_FILE = "/mnt/4TBSTORE/Dr_Kabuye4TB/processed_chunks/processed_chunk_20.csv"  # single CSV for final testing
TARGET_COLUMN = "category"
IGNORE_LABEL = -1
RANDOM_SEED = 42

# If you have multi-class classification, put all possible labels here, e.g. [-1, 0, 1].
# If you are doing binary classification and ignoring -1, use [0, 1].
ALL_CLASSES = np.array([0, 1])

# -------------------------------------------------------------
# HELPER FUNCTIONS
# -------------------------------------------------------------
def parse_space_separated_floats(embed_str):
    """
    If your llm_embeddings look like:
      "[-2.86e-01 -2.14e-01 2.51e-01 -1.58e-01]"
    with space-separated floats (no commas), parse them manually.
    If you have a valid Python list string, you can do literal_eval instead.
    """
    embed_str = embed_str.strip()
    embed_str = embed_str.strip("[]")
    tokens = embed_str.split()
    floats = list(map(float, tokens))
    return floats

def load_chunk(csv_path, parse_func=parse_space_separated_floats):
    """
    Load embeddings & labels from a single CSV file.
    Returns: X (np.ndarray), y (np.ndarray)
    """
    df = pd.read_csv(csv_path)
    if "llm_embeddings" not in df.columns or TARGET_COLUMN not in df.columns:
        return None, None

    # Convert embedding strings -> float arrays
    raw_embeds = df["llm_embeddings"].apply(lambda x: parse_func(str(x)))
    X = np.vstack(raw_embeds.values)
    y = df[TARGET_COLUMN].values
    return X, y

# -------------------------------------------------------------
# MAIN CHUNKED TRAINING
# -------------------------------------------------------------
def main():
    # 1) Initialize an SGDClassifier for partial fit
    # We'll treat it like logistic regression: loss="log_loss"
    clf = SGDClassifier(
        loss="log_loss",       # logistic regression
        max_iter=1,           # we'll do multiple partial_fit passes
        tol=None,             # no early stopping
        random_state=RANDOM_SEED
    )
    
    # 2) We may need to do an initial partial_fit with the classes= parameter
    #    so sklearn knows about all possible classes. This is only needed the very first time.
    #    We'll do that with a dummy array if we can, or the first chunk loaded.

    # Gather all training CSV files
    train_csvs = sorted(glob(os.path.join(TRAIN_DIR, "*.csv")))
    if not train_csvs:
        print(f"No training CSVs found in {TRAIN_DIR}. Exiting.")
        return

    # -- Optional: do an initial pass to define the classes
    #    We'll load just a small chunk from the first file to call partial_fit with classes=...
    init_file = train_csvs[0]
    X_init, y_init = load_chunk(init_file)
    if X_init is None:
        print(f"Could not read embeddings from {init_file}. Exiting.")
        return

    # Filter out -1 if ignoring
    mask_init = (y_init != IGNORE_LABEL)
    X_init = X_init[mask_init]
    y_init = y_init[mask_init]

    # If after filtering we have no rows, we skip to next file
    if len(y_init) == 0:
        print(f"First chunk has no valid rows after filter. Exiting or pick next file.")
        return

    # partial_fit once to initialize classes
    clf.partial_fit(X_init, y_init, classes=ALL_CLASSES)
    print(f"Initialized partial_fit with first chunk from {init_file} ...")

    # 3) Continue partial fit on the rest of the files (including the same file's remainder if you want)
    #    We'll skip the init rows or just do the entire file again with the partial_fit. It's up to you:
    #    For simplicity, let's re-fit on that chunk now (no big harm).
    clf.partial_fit(X_init, y_init)

    # If you want to skip the first file entirely from the loop below, do:
    # train_csvs = train_csvs[1:]

    # Now do partial fits on all training CSVs
    for csv_path in tqdm(train_csvs, desc="Training on chunks"):
        X_chunk, y_chunk = load_chunk(csv_path)
        if X_chunk is None:
            continue
        # Filter out -1
        mask_chunk = (y_chunk != IGNORE_LABEL)
        X_chunk = X_chunk[mask_chunk]
        y_chunk = y_chunk[mask_chunk]

        if len(y_chunk) == 0:
            continue

        # partial fit
        clf.partial_fit(X_chunk, y_chunk)

    print("Partial fit complete for all training chunks.")

    # 4) Load test CSV
    if not os.path.exists(TEST_FILE):
        print(f"Test file {TEST_FILE} not found. Cannot evaluate.")
        return

    X_test, y_test = load_chunk(TEST_FILE)
    if X_test is None:
        print(f"Could not load test data from {TEST_FILE}. No evaluation.")
        return
    # Filter out -1 if ignoring
    test_mask = (y_test != IGNORE_LABEL)
    X_test = X_test[test_mask]
    y_test = y_test[test_mask]
    if len(y_test) == 0:
        print("No valid test samples after filtering. Exiting.")
        return

    # 5) Evaluate
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="binary")
    rec = recall_score(y_test, y_pred, average="binary")
    f1 = f1_score(y_test, y_pred, average="binary")

    print("\n--- Final Evaluation on Test Set ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    main()


/tmp/ipykernel_234126/1985582278.py:48: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


KeyboardInterrupt: 